In [ ]:
# CELL 1: Install imdlib (Run this first!)
!pip install imdlib
!pip install xarray netcdf4
!pip install pandas numpy

print("✓ Installation complete! Now run the extraction code.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.9 MB/s eta 0:00:00
✓ Installation complete! Now run the extraction code.


In [ ]:
"""
IMD Data Extraction for 15 Stations
Date Range: 01-06-2018 to 30-09-2025
Output: Single Excel file with all stations vertically stacked
Columns: date, doy, rain, tmin, tmax, station_name, station_id
"""

import imdlib as imd
import pandas as pd
import numpy as np
import xarray as xr
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# INSTALL REQUIRED PACKAGES (RUN FIRST!)
# ============================================================================

!pip install imdlib -q
!pip install openpyxl -q

print("✅ Packages installed successfully!")

# ============================================================================
# 15 STATIONS (In Your Order)
# ============================================================================

STATIONS = [
    ('Anantwadi', 'Anantwadi', 19.8461, 77.8672, 'Maharashtra', 'Yavatmal'),
    ('Arni', 'Arni', 20.0619, 77.9486, 'Maharashtra', 'Yavatmal'),
    ('Khadka', 'Khadka', 19.9103, 78.1853, 'Maharashtra', 'Yavatmal'),
    ('KoliBk', 'Koli(Bk)', 20.0414, 78.1667, 'Maharashtra', 'Yavatmal'),
    ('Murli', 'Murli', 19.4681, 77.9861, 'Maharashtra', 'Yavatmal'),
    ('Sharad', 'Sharad', 19.9700, 78.3353, 'Maharashtra', 'Yavatmal'),
    ('Takali', 'Takali', 19.8442, 78.6008, 'Maharashtra', 'Yavatmal'),
    ('Bhamragad', 'Bhamragad', 19.4164, 80.5861, 'Maharashtra', 'Gadchiroli'),
    ('Hamdapur', 'Hamdapur', 20.6806, 78.8511, 'Maharashtra', 'Wardha'),
    ('Kunturla_Machkund', 'Kunturla_Machkund', 18.1992, 82.6497, 'Andhra Pradesh', 'Alluri Sitharama Raju'),
    ('Ankushapur', 'Ankushapur', 18.3528, 79.6381, 'Telangana', 'Bhupalpally'),
    ('Garimillapalli', 'Garimillapalli', 18.4469, 79.7047, 'Telangana', 'Nalgonda'),
    ('AllamVariGhanapur', 'AllamVariGhanapur', 18.3378, 80.4300, 'Telangana', 'Warangal'),
    ('Gandlapet', 'Gandlapet', 18.8211, 78.4381, 'Telangana', 'Nizamabad'),
    ('Valamuru_Pamuleru', 'Valamuru_Pamuleru', 17.6331, 81.6361, 'Andhra Pradesh', 'East Godavari'),
]


class MultiStationIMDExtractor:
    """
    Extract IMD data for multiple stations and create vertical Excel output
    """

    def __init__(self, stations_list, output_dir='/content/imd_data_output'):
        """Initialize extractor"""
        self.stations = pd.DataFrame(
            stations_list,
            columns=['station_code', 'station_name', 'latitude', 'longitude', 'state', 'district']
        )

        self.output_dir = output_dir
        self.data_dir = f'{output_dir}/imd_raw'
        self.processed_dir = f'{output_dir}/processed'

        # Setup directories
        for directory in [self.output_dir, self.data_dir, self.processed_dir]:
            os.makedirs(directory, exist_ok=True)

        print("="*80)
        print("🎯 MULTI-STATION IMD DATA EXTRACTOR")
        print("="*80)
        print(f"📍 Total Stations: {len(self.stations)}")
        print(f"📅 Date Range: 01-June-2018 to 30-Sep-2025")
        print(f"📁 Output: {output_dir}")
        print("="*80)

        # Show stations
        print(f"\n📋 Stations to extract (in order):")
        for idx, station in self.stations.iterrows():
            print(f"  Station {idx+1:2d}. {station['station_name']:20s} - Lat: {station['latitude']:.4f}, Lon: {station['longitude']:.4f}")
        print("="*80)

    def get_nearest_grid_point(self, lat, lon):
        """Get nearest IMD grid point (0.25° resolution)"""
        grid_lat = round(lat * 4) / 4
        grid_lon = round(lon * 4) / 4
        return grid_lat, grid_lon

    def download_variable(self, variable, start_year, end_year):
        """Download IMD data for one variable"""
        print(f"\n{'='*80}")
        print(f"📥 DOWNLOADING {variable.upper()}: {start_year}-{end_year}")
        print(f"{'='*80}")

        try:
            print(f"⏳ Downloading... (may take 2-3 minutes)")
            imd.get_data(
                variable,
                start_year,
                end_year,
                fn_format='yearwise',
                file_dir=self.data_dir
            )
            print(f"✅ Download complete for {variable}!")
            return True

        except Exception as e:
            print(f"❌ Download failed: {e}")
            return False

    def load_variable_data(self, variable, start_year, end_year):
        """Load downloaded IMD data"""
        print(f"\n📂 Loading {variable} data...")

        try:
            # Load using imdlib
            imd_obj = imd.open_data(
                variable,
                start_year,
                end_year,
                fn_format='yearwise',
                file_dir=self.data_dir
            )

            # Convert to xarray Dataset
            data = imd_obj.get_xarray()
            print(f"✅ Data loaded successfully!")

            return data

        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return None

    def extract_station(self, data, variable, station_row, start_year, end_year, station_id):
        """Extract data for one station"""
        try:
            station_lat = station_row['latitude']
            station_lon = station_row['longitude']
            station_name = station_row['station_name']
            station_code = station_row['station_code']

            # Get nearest grid point
            grid_lat, grid_lon = self.get_nearest_grid_point(station_lat, station_lon)

            # Extract station data
            station_data = data.sel(lat=grid_lat, lon=grid_lon, method='nearest')

            # Convert to DataFrame
            df = station_data[variable].to_dataframe().reset_index()
            df = df[['time', variable]].rename(columns={'time': 'date'})

            # Remove missing values (-999.0 in IMD data)
            df = df[df[variable] != -999.0].copy()

            # Format date
            df['date'] = pd.to_datetime(df['date'])

            # Filter date range: 01-June-2018 to 30-Sep-2025
            start_date = '2018-06-01'
            end_date = '2025-09-30'
            df = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()

            # Add metadata
            df['station_name'] = station_name
            df['station_id'] = f'Station {station_id}'

            if len(df) == 0:
                return None

            return df

        except Exception as e:
            print(f"      Error: {e}")
            return None

    def process_all_variables(self, start_year=2018, end_year=2025):
        """Process all variables for all stations"""
        print(f"\n{'#'*80}")
        print(f"🚀 STARTING EXTRACTION")
        print(f"{'#'*80}\n")

        # Storage for all station data
        all_station_data = []

        # Download and load data for each variable
        variables = ['rain', 'tmin', 'tmax']
        data_dict = {}

        for var in variables:
            print(f"\n{'='*80}")
            print(f"📊 PROCESSING {var.upper()}")
            print(f"{'='*80}")

            # Download
            if not self.download_variable(var, start_year, end_year):
                print(f"❌ Skipping {var}")
                continue

            # Load
            data = self.load_variable_data(var, start_year, end_year)
            if data is None:
                print(f"❌ Skipping {var}")
                continue

            data_dict[var] = data

        # Extract data for each station
        print(f"\n{'='*80}")
        print(f"📍 EXTRACTING ALL STATIONS")
        print(f"{'='*80}\n")

        for idx, station in self.stations.iterrows():
            station_id = idx + 1
            station_name = station['station_name']

            print(f"Station {station_id:2d}. {station_name:20s} ", end='')

            # Extract all variables for this station
            station_df_list = []

            for var in variables:
                if var not in data_dict:
                    continue

                df = self.extract_station(
                    data_dict[var],
                    var,
                    station,
                    start_year,
                    end_year,
                    station_id
                )

                if df is not None:
                    station_df_list.append(df[['date', var, 'station_name', 'station_id']])

            # Merge all variables for this station
            if len(station_df_list) == 3:
                # Merge rain, tmin, tmax
                merged = station_df_list[0].merge(
                    station_df_list[1][['date', 'tmin']],
                    on='date',
                    how='outer'
                )
                merged = merged.merge(
                    station_df_list[2][['date', 'tmax']],
                    on='date',
                    how='outer'
                )

                # Add DOY (Day of Year)
                merged['doy'] = merged['date'].dt.dayofyear

                # Reorder columns: date, doy, rain, tmin, tmax, station_name, station_id
                merged = merged[['date', 'doy', 'rain', 'tmin', 'tmax', 'station_name', 'station_id']]

                # Sort by date
                merged = merged.sort_values('date').reset_index(drop=True)

                all_station_data.append(merged)

                print(f"✅ {len(merged):,} records")
            else:
                print(f"❌ Failed")

        # Combine all stations vertically
        print(f"\n{'='*80}")
        print(f"🔗 COMBINING ALL STATIONS")
        print(f"{'='*80}")

        if not all_station_data:
            print("❌ No data extracted!")
            return None

        # Concatenate all stations (one after another)
        final_df = pd.concat(all_station_data, ignore_index=True)

        print(f"\n✅ Combined {len(all_station_data)} stations")
        print(f"📊 Total records: {len(final_df):,}")
        print(f"📅 Date range: {final_df['date'].min()} to {final_df['date'].max()}")

        return final_df

    def save_to_excel(self, df, filename='imd_data_all_stations.xlsx'):
        """Save DataFrame to Excel"""
        print(f"\n{'='*80}")
        print(f"💾 SAVING TO EXCEL")
        print(f"{'='*80}")

        output_path = f"{self.output_dir}/{filename}"

        try:
            # Save to Excel
            df.to_excel(output_path, index=False, sheet_name='IMD_Data')

            print(f"\n✅ Excel file saved: {output_path}")
            print(f"📊 File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

            # Show sample
            print(f"\n📋 Sample data (first 10 rows):")
            print(df.head(10).to_string(index=False))

            print(f"\n📋 Sample data (last 10 rows):")
            print(df.tail(10).to_string(index=False))

            # Statistics
            print(f"\n📈 STATISTICS:")
            print(f"   Total rows: {len(df):,}")
            print(f"   Stations: {df['station_id'].nunique()}")
            print(f"   Date range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
            print(f"   Total days: {df['date'].nunique()}")

            # Per station stats
            print(f"\n📊 Records per station:")
            for station_id in df['station_id'].unique():
                station_data = df[df['station_id'] == station_id]
                station_name = station_data['station_name'].iloc[0]
                print(f"   {station_id}: {station_name:20s} - {len(station_data):,} records")

            return output_path

        except Exception as e:
            print(f"❌ Error saving Excel: {e}")
            return None

    def run(self, start_year=2018, end_year=2025):
        """Main execution function"""
        # Extract all data
        final_df = self.process_all_variables(start_year, end_year)

        if final_df is None:
            print("\n❌ Extraction failed!")
            return None

        # Save to Excel
        excel_path = self.save_to_excel(final_df, 'imd_data_all_stations_2018_2025.xlsx')

        return excel_path


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*80)
    print("🎯 IMD DATA EXTRACTION - 15 STATIONS")
    print("   Date Range: 01-June-2018 to 30-Sep-2025")
    print("   Output Format: Vertical Excel (all stations stacked)")
    print("="*80)

    # Initialize extractor
    extractor = MultiStationIMDExtractor(
        STATIONS,
        output_dir='/content/imd_data_output'
    )

    # Run extraction
    print("\n⚠️  This will take 5-10 minutes")
    print("   Please wait...\n")

    excel_path = extractor.run(
        start_year=2018,
        end_year=2025
    )

    if excel_path:
        print("\n" + "="*80)
        print("✅ ALL DONE!")
        print("="*80)
        print(f"\n🎉 Excel file created: {excel_path}")
        print(f"\n📥 Download the file from the left sidebar (Files icon)")
        print(f"   Location: {excel_path}")

        print(f"\n📋 File Structure:")
        print(f"   Columns: date | doy | rain | tmin | tmax | station_name | station_id")
        print(f"   Format: All 15 stations stacked vertically")
        print(f"   Order: Same as your station list (Station 1 to Station 15)")

        print(f"\n💡 How to use in Python:")
        print(f"""
import pandas as pd

# Load the Excel file
df = pd.read_excel('{excel_path}')

# Filter by station
station1_data = df[df['station_id'] == 'Station 1']
station2_data = df[df['station_id'] == 'Station 2']

# Or filter by station name
anantwadi_data = df[df['station_name'] == 'Anantwadi']

# Filter by date range
june_data = df[df['date'].dt.month == 6]
        """)
    else:
        print("\n❌ Extraction failed! Please check the errors above.")




✅ Packages installed successfully!

🎯 IMD DATA EXTRACTION - 15 STATIONS
   Date Range: 01-June-2018 to 30-Sep-2025
   Output Format: Vertical Excel (all stations stacked)
🎯 MULTI-STATION IMD DATA EXTRACTOR
📍 Total Stations: 15
📅 Date Range: 01-June-2018 to 30-Sep-2025
📁 Output: /content/imd_data_output

📋 Stations to extract (in order):
  Station  1. Anantwadi            - Lat: 19.8461, Lon: 77.8672
  Station  2. Arni                 - Lat: 20.0619, Lon: 77.9486
  Station  3. Khadka               - Lat: 19.9103, Lon: 78.1853
  Station  4. Koli(Bk)             - Lat: 20.0414, Lon: 78.1667
  Station  5. Murli                - Lat: 19.4681, Lon: 77.9861
  Station  6. Sharad               - Lat: 19.9700, Lon: 78.3353
  Station  7. Takali               - Lat: 19.8442, Lon: 78.6008
  Station  8. Bhamragad            - Lat: 19.4164, Lon: 80.5861
  Station  9. Hamdapur             - Lat: 20.6806, Lon: 78.8511
  Station 10. Kunturla_Machkund    - Lat: 18.1992, Lon: 82.6497
  Station 11. Ankusha